# DS555 — HATCO Project Analysis
## HATCO Customer Satisfaction Study
**Data:** `data/HATCO_clean.csv` — 100 observations, 14 variables  
**Course:** DS555 Multivariate Statistical Analysis

---


## Step 0 — Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("../data/HATCO_clean.csv")
print("Shape:", df.shape)
df.head()


---
## Section 3A — One-Way ANOVA

**Research Question:** Does the mean satisfaction level differ between Small and Large firms?

| | |
|---|---|
| **IV (Independent)** | `firm_size` — 0 = Small, 1 = Large |
| **DV (Dependent)** | `satisfaction` — scale 0–10 |
| **H₀** | µ_small = µ_large |
| **H₁** | µ_small ≠ µ_large |
| **α** | 0.05 |


In [ ]:
# ─────────────────────────────────────────────
#  ONE-WAY ANOVA — Satisfaction by Firm Size
# ─────────────────────────────────────────────

# Split groups
small = df[df["firm_size"] == 0]["satisfaction"]
large = df[df["firm_size"] == 1]["satisfaction"]

print(f"Small firms  (n={len(small)}): mean={small.mean():.2f}, std={small.std():.2f}")
print(f"Large firms  (n={len(large)}): mean={large.mean():.2f}, std={large.std():.2f}")

# Step 1: Normality — Shapiro-Wilk
print("\n--- Step 1: Normality Test (Shapiro-Wilk) ---")
stat_s, p_s = stats.shapiro(small)
stat_l, p_l = stats.shapiro(large)
print(f"Small: W={stat_s:.4f}, p={p_s:.4f}  -> {'Normal' if p_s > 0.05 else 'NOT normal'}")
print(f"Large: W={stat_l:.4f}, p={p_l:.4f}  -> {'Normal' if p_l > 0.05 else 'NOT normal'}")

# Step 2: Homogeneity — Levene
print("\n--- Step 2: Levene Test (Homogeneity of Variance) ---")
lev_stat, lev_p = stats.levene(small, large)
print(f"Levene: F={lev_stat:.4f}, p={lev_p:.4f}  -> {'Equal variances' if lev_p > 0.05 else 'Unequal variances'}")

# Step 3: One-Way ANOVA
print("\n--- Step 3: One-Way ANOVA ---")
f_stat, p_val = stats.f_oneway(small, large)
print(f"F = {f_stat:.4f},  p = {p_val:.4f}")
if p_val < 0.05:
    print("Decision: REJECT H0 -> Significant difference in satisfaction between firm sizes")
else:
    print("Decision: FAIL TO REJECT H0 -> No significant difference")

# Boxplot
fig, ax = plt.subplots(figsize=(6, 4))
ax.boxplot([small, large], labels=["Small Firm", "Large Firm"])
ax.set_title("Figure 15: Satisfaction Level by Firm Size")
ax.set_ylabel("Satisfaction (0-10)")
plt.tight_layout()
plt.savefig("../plots/Figure_15_anova_boxplot.png", dpi=120)
plt.show()


---
### Results & Interpretation

#### Step 1 — Normality Test (Shapiro-Wilk)

| Group | n | W | p-value | Decision |
|---|---|---|---|---|
| Small firms | 60 | 0.9836 | 0.5964 | Normal ✓ |
| Large firms | 40 | 0.9363 | **0.0260** | Not normal |

> The Large firms group shows a slight deviation from normality (p = 0.026 < 0.05). However, both groups have n ≥ 30, so the **Central Limit Theorem (CLT)** ensures the ANOVA remains valid and robust to this violation.

#### Step 2 — Levene's Test (Homogeneity of Variance)

| F | p-value | Decision |
|---|---|---|
| 0.1942 | 0.6604 | Equal variances ✓ |

> Since p = 0.66 > 0.05, we **fail to reject** the null hypothesis of equal variances. The assumption of homogeneity of variance is satisfied, making the standard ANOVA appropriate.

#### Step 3 — One-Way ANOVA

| | Small Firms (n=60) | Large Firms (n=40) |
|---|---|---|
| **Mean** | 5.09 | 4.29 |
| **Std** | 0.75 | 0.78 |

| F-statistic | p-value | Decision |
|---|---|---|
| **25.8067** | **< 0.001** | **Reject H₀** |

> **Conclusion:** There is a statistically significant difference in satisfaction levels between Small and Large firms (F = 25.81, p < 0.001). Small firms report a notably higher mean satisfaction (5.09) compared to Large firms (4.29), with a mean difference of **0.80 points** on a 0–10 scale.

> This suggests that smaller purchasing firms perceive HATCO's service more favorably than larger ones — possibly because smaller firms have simpler requirements that HATCO meets more consistently.

> **Note on normality:** Post-hoc tests (Tukey HSD) are not needed here since there are only 2 groups. The significant F-test is sufficient to conclude a group difference.
